# Lab 14 (cont.): Deal profile and the screening memo

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2 | LLM mode: replay


In [2]:
import re, yaml
LARGE = llm.P["llm"]["models"]["large"]
key = yaml.safe_load(open("data/dataroom/deal_a/answer_key.yaml"))
TRUTH = {"revenue_FY2025": 41.2e6, "revenue_FY2024": 33.07e6, "revenue_growth_FY2025": key["traps"][0]["true_value"], "nrr_FY2025": 1.09,
         "top1_customer_share_FY2025": 0.145, "adjusted_ebitda_FY2025": 9.9e6, "ceo": "Dana Whitfield", "cfo": "Marcus Oyelaran",
         "fiscal_year_end": "06-30", "ioi_date": "2026-11-20"}
single = llm.structured(schemas.DealProfile, prompts.profile_single_messages(sd.cim_text()), LARGE)
def from_profile(p):
    k = {f"{x.name}_{x.period}": x.value for x in p.revenue_history + p.ebitda_history + p.kpis}
    names = {m.split(",")[1].strip(): m.split(",")[0] for m in p.management}
    d = re.search(r"(\w+ \d{1,2}, \d{4})", p.process_terms)
    from datetime import datetime
    return {"revenue_FY2025": k.get("revenue_FY2025"), "revenue_FY2024": k.get("revenue_FY2024"), "revenue_growth_FY2025": k.get("revenue_growth_FY2025"),
            "nrr_FY2025": k.get("nrr_FY2025"), "top1_customer_share_FY2025": p.top_customers[0].revenue_share if p.top_customers else None,
            "adjusted_ebitda_FY2025": k.get("adjusted_ebitda_FY2025"), "ceo": names.get("Chief Executive Officer"), "cfo": names.get("Chief Financial Officer"),
            "fiscal_year_end": None, "ioi_date": datetime.strptime(d[1], "%B %d, %Y").date().isoformat() if d else None}
pred_single = from_profile(single)

In [3]:
# Staged: seller facts (tier 3) where they exist, then section-by-section reads of the CIM records
F = {f'{f["metric"]}_{f["period"]}': f["value"] for f in sd.seller_facts()}
txt = {r["block_id"]: r["text"] for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # unique only within a document
mgmt = re.findall(r"(Chief \w+ Officer) ([A-Z]\w+ [A-Z]\w+)", txt["p0005_b001"])
pred_staged = {**{k: F.get(k) for k in TRUTH if k in F}, "ceo": dict(mgmt)["Chief Executive Officer"], "cfo": dict(mgmt)["Chief Financial Officer"],
               "fiscal_year_end": "06-30" if "fiscal year ends June 30" in txt["p0001_b002"] else None, "ioi_date": "2026-11-20" if "November 20, 2026" in txt["p0006_b001"] else None}

In [4]:
# From the slide: Single prompt or staged? Measure it, with an interval
import random

def field_correct(pred, key, tol=0.005):   # Kestrel key
    out = []
    for field, true in key.items():
        p = pred.get(field)
        if isinstance(true, (int, float)) and p is not None:
            out.append(int(abs(p - true) <= tol * abs(true)))
        else:
            out.append(int(p == true))
    return out

def bootstrap_ci(correct, n=2000, seed=7):
    r = random.Random(seed)
    means = sorted(sum(r.choices(correct, k=len(correct)))
                   / len(correct) for _ in range(n))
    return means[int(0.025 * n)], means[int(0.975 * n) - 1]

In [5]:
for name, pred in (("single prompt", pred_single), ("staged", pred_staged)):
    correct = field_correct(pred, TRUTH)
    lo, hi = bootstrap_ci(correct)
    print(f"{name:14s} {sum(correct)}/{len(correct)} fields correct, 95% CI [{lo:.2f}, {hi:.2f}]")

single prompt  6/10 fields correct, 95% CI [0.30, 0.90]
staged         10/10 fields correct, 95% CI [1.00, 1.00]


In [6]:
SECTIONS = ["Company Overview", "Business Model and Go-to-Market", "Financial Profile", "Customers and Concentration",
            "Management and Organization", "Process and Valuation Context", "Key Risks and Diligence Questions", "Disclosure Gaps"]
body = {"Company Overview": txt["p0001_b001"],
        "Financial Profile": f"FY2025 revenue ${F['revenue_FY2025']/1e6:.1f} million (per seller financials), growth {F['revenue_growth_FY2025']:.1%}; the CIM claims 32% (unverified).",
        "Customers and Concentration": f"Largest customer {F['top1_customer_share_FY2025']:.1%} of FY2025 revenue (per seller financials); the CIM claims no customer above 8%.",
        "Management and Organization": f"CEO {pred_staged['ceo']}; CFO {pred_staged['cfo']}.",
        "Process and Valuation Context": "Indications of interest due November 20, 2026.",
        "Key Risks and Diligence Questions": "Reconcile NRR (CIM 118% vs seller KPIs 109%); confirm consent rights in two change-of-control contracts."}
memo = "\n\n".join(f"## {s}\n{body.get(s, 'Not disclosed.')}" for s in SECTIONS)
Path("outputs/memos").mkdir(parents=True, exist_ok=True); Path("outputs/memos/deal_a_structured.md").write_text(memo)
print(memo[:600])

## Company Overview
Wren Freight Software, Inc. ("Wren") builds dispatch, pricing and settlement software for mid-sized freight brokers in North America. Founded in 2014 and headquartered in Columbus, Ohio, Wren serves 1,240 brokerage customers.

## Business Model and Go-to-Market
Not disclosed.

## Financial Profile
FY2025 revenue $41.2 million (per seller financials), growth 24.6%; the CIM claims 32% (unverified).

## Customers and Concentration
Largest customer 14.5% of FY2025 revenue (per seller financials); the CIM claims no customer above 8%.

## Management and Organization
CEO Dana Whit


**Exercise 19.** With only ten fields, the intervals are wide. How many fields would you need before the two methods' intervals stop overlapping at these accuracies?

In [7]:
import math
import pandas as pd

def wilson_interval(k, n, z=1.96):
    """95% Wilson interval from integer successes and trials."""
    if not isinstance(k, int) or not isinstance(n, int) or n <= 0 or not 0 <= k <= n:
        raise ValueError("Require integers 0 <= k <= n and n > 0.")
    p = k / n
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denominator
    return center - margin, center + margin

single_correct = sum(field_correct(pred_single, TRUTH))
staged_correct = sum(field_correct(pred_staged, TRUTH))
assert (single_correct, staged_correct, len(TRUTH)) == (6, 10, 10)
a10 = wilson_interval(single_correct, len(TRUTH))
b10 = wilson_interval(staged_correct, len(TRUTH))
print("6/10 Wilson interval:", tuple(round(x, 3) for x in a10))
print("10/10 Wilson interval:", tuple(round(x, 3) for x in b10))
print("Intervals overlap at n=10:", a10[1] >= b10[0])
assert a10[1] >= b10[0]
assert b10[0] < 1  # Unlike the degenerate empirical bootstrap interval.

# Match the handout's integer-count convention: round(0.6 * n) vs n.
# Compare unrounded intervals; round only for display.
rows = []
for fields in range(1, 60):
    successes = round(0.6 * fields)
    single_ci = wilson_interval(successes, fields)
    staged_ci = wilson_interval(fields, fields)
    separate = single_ci[1] < staged_ci[0]
    rows.append({
        "fields": fields, "single_correct": successes,
        "single_accuracy": successes / fields,
        "single_interval": tuple(round(x, 3) for x in single_ci),
        "staged_interval": tuple(round(x, 3) for x in staged_ci),
        "non_overlapping": separate,
    })
    if separate:
        break
search_results = pd.DataFrame(rows)
assert search_results.iloc[-1]["non_overlapping"]
assert not search_results.iloc[:-1]["non_overlapping"].any()
minimum_fields = int(search_results.iloc[-1]["fields"])
assert minimum_fields == 17
print("Smallest n under the handout's rounded-count convention:", minimum_fields)
display(search_results[search_results["fields"] >= 10])
print("At n=17: 10/17 single-prompt successes versus 17/17 staged successes.")


6/10 Wilson interval: (0.313, 0.832)
10/10 Wilson interval: (0.722, 1.0)
Intervals overlap at n=10: True
Smallest n under the handout's rounded-count convention: 17


,fields,single_correct,single_accuracy,single_interval,staged_interval,non_overlapping
9,10,6,0.600000,"(0.313, 0.832)","(0.722, 1.0)",False
10,11,7,0.636364,"(0.354, 0.848)","(0.741, 1.0)",False
11,12,7,0.583333,"(0.32, 0.807)","(0.757, 1.0)",False
12,13,8,0.615385,"(0.355, 0.823)","(0.772, 1.0)",False
13,14,8,0.571429,"(0.326, 0.786)","(0.785, 1.0)",False
14,15,9,0.600000,"(0.357, 0.802)","(0.796, 1.0)",False
15,16,10,0.625000,"(0.386, 0.815)","(0.806, 1.0)",False
16,17,10,0.588235,"(0.36, 0.784)","(0.816, 1.0)",True


At n=17: 10/17 single-prompt successes versus 17/17 staged successes.


**Why does the staged bootstrap interval equal [1.00, 1.00]?** All ten observed correctness indicators are 1. Every resample therefore also contains only 1s, so every resampled accuracy is 100%. This describes the resampling distribution of this tiny observed sample; it does not establish perfect population accuracy.

**Wilson comparison.** At n=10, the intervals are approximately [0.313, 0.832] for 6/10 and [0.722, 1.000] for 10/10. They overlap. Checking every integer n finds the first separation at **17 fields**, under the handout's convention `round(0.6 * n)` versus `n` successes: [0.360, 0.784] for 10/17 and [0.816, 1.000] for 17/17. The earlier answer of 20 was only the first sufficient value among a sparse list of tested sample sizes.

**What this means.** At n=17 the rounded single-prompt accuracy is 10/17 (about 58.8%), not exactly 60%. Thus 17 is conditional on the specified rounding convention and assumed future outcomes. Non-overlapping marginal intervals are a classroom comparison, not a formal power calculation or paired significance test. Evaluate additional independent fields, ideally across deals; duplicating the same ten observations adds no evidence.

**Fairness of the vendor comparison.** Selecting only convenient fields can inflate accuracy or favor a method. Evaluating every data point avoids that selection but may overweight repeated or easy values and requires an explicit definition of a data point. Predefine the field inventory, inclusion rules, and scoring before seeing predictions, and report missing or unsupported fields consistently.
